In [ ]:
!pip install catboost

# Imports

In [ ]:
import os
import joblib
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from catboost import CatBoostClassifier
from sklearn.metrics import matthews_corrcoef

from sklearn.model_selection import(
    StratifiedKFold,
    StratifiedGroupKFold,
    RandomizedSearchCV,
    cross_val_predict
)

from utils.models import (
    get_models,
    get_base_models,
    get_age_baseline
)

# from utils.nested_cv import run_outer_fold_loop
from utils.evaluation import evaluate_model, evaluate_outer_fold

from utils.optimization import(
    get_param_dist,
    get_search_iter,
    # tune_model
)

from utils.pipeline import(
    create_pipeline,
    save_fold_predictions,
    # select_best_threshold
)

from itertools import chain
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Dataset 1: CKD Dataset

In [ ]:
ds1 = pd.read_csv("/content/CKD_preprocessed.csv")

ds1_X = ds1.drop("d_status", axis=1)
ds1_y = ds1["d_status"]

# Cross-validation objects

In [ ]:
#### Dataset 1 - CKD Dataset ####
outer_cv_d1 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

inner_cv_d1 = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)


# Load all models

In [ ]:
models = get_models()
base_models = get_base_models()

# Find best parameters and best pipeline

In [ ]:
def tune_model(model_name, pipeline, X_train, y_train, inner_cv, scoring, groups=None):

  if model_name in ["Majority Dummy", "Stratified Dummy"]:
    pipeline.fit(X_train, y_train)
    return pipeline, {}, None

  ### RandomizedSearchCV ###
  random_search = RandomizedSearchCV(

      estimator=pipeline,
      param_distributions=get_param_dist(model_name),
      n_iter=get_search_iter(model_name),
      scoring=scoring,
      cv=inner_cv,
      random_state=RANDOM_STATE,
      n_jobs=-1,
      refit=True
    )

  random_search.fit(
          X_train,
          y_train,
          groups=groups
  )

  ### Best Estimator ###
  best_pipeline = random_search.best_estimator_
  best_params = random_search.best_params_
  best_score = random_search.best_score_

  return (best_pipeline, best_params, best_score)

# Find best threshold

In [ ]:
def select_best_threshold(model_name, best_param, X_train, y_train, inner_cv, groups=None):
  ### Generate inner out-of-fold probabilities ###
  inner_probabilities = np.zeros(len(y_train))

  # Generate inner folds
  if groups is None:
      splits = inner_cv.split(X_train, y_train)
  else:
      splits = inner_cv.split(X_train, y_train, groups)

  for inner_train_idx, inner_valid_idx in splits:

        # Split inner fold
        X_inner_train = X_train.iloc[inner_train_idx]
        y_inner_train = y_train.iloc[inner_train_idx]

        X_inner_valid = X_train.iloc[inner_valid_idx]

        # Create a fresh pipeline
        if model_name in ["Majority Dummy", "Stratified Dummy"]:
          model = get_base_models()[model_name]
        else:
          model = get_models()[model_name]
        pipeline = create_pipeline(model_name, model)

        # Apply tuned parameters
        pipeline.set_params(**best_param)

        # Train
        pipeline.fit(X_inner_train, y_inner_train)

        # Predict probabilities
        inner_probabilities[inner_valid_idx] = (
            pipeline.predict_proba(X_inner_valid)[:, 1]
        )

   ##### Find best threshold #####
  thresholds=np.arange(0.01, 1.00, 0.01)
  best_threshold = 0.5
  best_mcc = -1
  for threshold in thresholds:
          y_pred = (inner_probabilities >= threshold).astype(int)
          mcc = matthews_corrcoef(y_train, y_pred)

          if mcc > best_mcc:
              best_mcc = mcc
              best_threshold = threshold

  return best_threshold, best_mcc

# Evaluate fold

In [ ]:
def evaluate_outer_fold(estimator, X_test, y_test, threshold):
  test_probabilities = estimator.predict_proba(
        X_test
    )[:, 1]

  test_predictions = (
        test_probabilities >= threshold
    ).astype(int)

  ### Evaluate fold ###
  fold_metrics = evaluate_model(

        y_test= y_test,
        predictions= test_predictions,
        probabilities= test_probabilities

    )
  return fold_metrics, test_predictions, test_probabilities

# Outer fold for loop

In [ ]:
def run_outer_fold_loop(experiment1_ds, model_name, model, X, y, outer_cv, inner_cv,
                        scoring, groups=None, base_name=None):
  for fold, (train_idx, test_idx) in enumerate(
            outer_cv.split(X, y, groups),
            start=1):

        ################# Create train/test split #################
        X_train = X.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()

        if groups is None:
          group_train = None
        else:
          group_train = groups.iloc[train_idx]

        ################# Build pipeline #################
        pipeline = create_pipeline(
            model_name=model_name,
            estimator=model
        )

        ################# Get best pipeline and parameters #################
        best_pipeline, best_params, best_score = tune_model(
            model_name, pipeline, X_train, y_train,
            inner_cv, scoring, group_train)

        if base_name is None:
          base_name = model_name

        base_name = base_name.replace(" ","_")

        experiment1_ds[base_name]["best_params"].append(
            best_params
        )

        ################# Find best threshold #################
        threshold, best_inner_mcc = select_best_threshold(model_name, best_params, X_train, y_train,
                                                          inner_cv, group_train)
        experiment1_ds[base_name]["thresholds"].append(
            threshold
        )
        experiment1_ds[base_name]["best_inner_mcc"].append(
            best_inner_mcc
        )

        ################# Retrain on the outer training fold #################
        best_pipeline.fit(X_train, y_train)

        ################# Evaluate fold #################
        fold_metrics, y_pred, y_prob = evaluate_outer_fold(
            best_pipeline, X_test, y_test, threshold
        )

        fold_metrics['Fold'] = fold
        fold_metrics['Threshold'] = threshold
        experiment1_ds[base_name]["fold_metrics"].append(
            fold_metrics
        )

        ################# Save fold path #################
        fold_path = save_fold_predictions(FOLD_DIR, model_name, fold, best_pipeline, best_params, threshold,
                                          train_idx, test_idx, best_score, y_test, y_prob, y_pred)
        experiment1_ds[base_name]["saved_folds"].append(
            fold_path
        )

        ################# Save trained model #################
        model_dir = os.path.join(
          MODEL_DIR,
          base_name
        )

        os.makedirs(
            model_dir,
            exist_ok=True
        )
        model_path = os.path.join(model_dir, f"{base_name}_fold{fold}.joblib")

        joblib.dump(best_pipeline, model_path)
        experiment1_ds[base_name]["saved_models"].append(
            model_path
        )


# Output folders for CKD dataset

In [ ]:
OUTPUT_DIR = "/content/dataset1"

MODEL_DIR = os.path.join(OUTPUT_DIR, "saved_models")
CSV_DIR = os.path.join(OUTPUT_DIR, "csv")
FOLD_DIR = os.path.join(OUTPUT_DIR, "fold_predictions")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(FOLD_DIR, exist_ok=True)

# Experiment 1: CKD Dataset

---



In [ ]:
experiment1_ds1 = {}
result_rows = []

for model_name, model in chain(base_models.items(), models.items()):
  # print("=" * 70)
  print(model_name)
  # print("=" * 70)

  base_name = model_name.replace(" ","_")

  experiment1_ds1[base_name] = {
      "fold_metrics": [],
      "best_params": [],
      "thresholds": [],
      "best_inner_mcc": [],
      "saved_folds": [],
      "saved_models": []
  }

  ################# Outer fold loop #################
  run_outer_fold_loop(experiment1_ds1, model_name, model, ds1_X, ds1_y,
                      outer_cv_d1, inner_cv_d1, scoring="roc_auc")

  ################# Add aggregation #################
  fold_df = pd.DataFrame(
        experiment1_ds1[base_name]["fold_metrics"]

  )
  experiment1_ds1[base_name]["fold_results"] = fold_df

  # Columns that should be averaged
  metric_columns = [
      "ROC AUC",
      "PR AUC",
      "F1",
      "MCC",
      "Sensitivity",
      "Specificity",
      "Balanced Accuracy",
      "Accuracy",
      "Brier Score"
  ]

  ################# Calculate mean and std for the df #################
  metric_df = fold_df[metric_columns]
  mean = metric_df.mean()
  std = metric_df.std()

  experiment1_ds1[base_name]["mean"] = (
        mean.to_dict()
  )
  experiment1_ds1[base_name]["std"] = (
            std.to_dict()
  )

  ################# save fold results and best params #################
  csv_dir = os.path.join(
        CSV_DIR,
        base_name
  )

  os.makedirs(
        csv_dir,
        exist_ok=True
  )
  fold_df.to_csv(
      os.path.join(
          csv_dir,
          f"{base_name}_fold_results.csv"
      ),
      index=False,
      encoding='utf-8-sig'
  )

  params_df = pd.DataFrame(
      experiment1_ds1[base_name]["best_params"]
  )

  params_df.insert(0, "Fold", range(1, 6))

  params_df.to_csv(
      os.path.join(
          csv_dir,
          f"{base_name}_best_parameters.csv"
      ),
      index=False,
      encoding='utf-8-sig'
  )

  ################# Sort and save final result for CKD dataset #################
  result_dict = {}
  result_dict["Model"] = model_name
  for metric in metric_columns:
    result_dict[metric] = f"{mean[metric]:.4f} ± {std[metric]:.4f}"

  result_rows.append(result_dict)

################# Summary #################
result_df = pd.DataFrame(result_rows)
print("=" * 70)
print("Experiment1 CKD Dataset Result")
print("=" * 70)
print(result_df)

result_df = result_df.sort_values("ROC AUC", ascending=False)
result_df.to_csv(
    os.path.join(
        CSV_DIR,
        "experiment1_dataset1_result.csv"
    ),
    index=False,
    encoding='utf-8-sig'
)

joblib.dump(
    experiment1_ds1,
    os.path.join(OUTPUT_DIR, "experiment1_dataset1.joblib")
)

##### load joblib file #####
# experiment1_ds1 = joblib.load(
#     os.path.join(OUTPUT_DIR, "experiment1_dataset1.joblib")
# )

Majority Dummy
Stratified Dummy
Logistic Regression
Decision Tree
Random Forest
XGBoost
LightGBM
CatBoost
SVM
MLP
Experiment1 CKD Dataset Result
                 Model          ROC AUC           PR AUC               F1  \
0       Majority Dummy  0.5000 ± 0.0000  0.6289 ± 0.0059  0.7722 ± 0.0045   
1     Stratified Dummy  0.5374 ± 0.0337  0.6475 ± 0.0201  0.6761 ± 0.0255   
2  Logistic Regression  0.9714 ± 0.0209  0.9831 ± 0.0122  0.9261 ± 0.0319   
3        Decision Tree  0.9362 ± 0.0236  0.9421 ± 0.0218  0.9082 ± 0.0262   
4        Random Forest  0.9717 ± 0.0217  0.9833 ± 0.0121  0.9295 ± 0.0354   
5              XGBoost  0.9634 ± 0.0234  0.9771 ± 0.0163  0.9132 ± 0.0358   
6             LightGBM  0.9692 ± 0.0193  0.9813 ± 0.0118  0.9288 ± 0.0259   
7             CatBoost  0.9671 ± 0.0209  0.9798 ± 0.0139  0.9249 ± 0.0301   
8                  SVM  0.3982 ± 0.5213  0.6479 ± 0.3038  0.8377 ± 0.0878   
9                  MLP  0.9677 ± 0.0265  0.9810 ± 0.0146  0.9143 ± 0.0452   

       

['/content/dataset1/experiment1_dataset1.joblib']

In [ ]:
### for dataset 1 ###
# for fold, (train_idx, test_idx) in enumerate(
  #           outer_cv_d1.split(ds1_X, ds1_y),
  #           start=1):

  #       ################# Create train/test split #################
  #       ds1_X_train = ds1_X.iloc[train_idx].copy()
  #       ds1_X_test = ds1_X.iloc[test_idx].copy()

  #       ds1_y_train = ds1_y.iloc[train_idx].copy()
  #       ds1_y_test = ds1_y.iloc[test_idx].copy()

  #       ################# Build pipeline #################
  #       pipeline = create_pipeline(
  #           model_name=model_name,
  #           estimator=model
  #       )

  #       ################# Get best pipeline and parameters #################
  #       best_pipeline, best_params = tune_model(
  #           model_name, pipeline, ds1_X_train, ds1_y_train,
  #           inner_cv_d1, "roc_auc")

  #       experiment1_ds1[model_name]["best_params"].append(
  #           best_params
  #       )

  #       ################# Find best threshold #################
  #       threshold, best_inner_mcc = select_best_threshold(best_pipeline, ds1_X_train, ds1_y_train, inner_cv_d1)
  #       experiment1_ds1[model_name]["thresholds"].append(
  #           threshold
  #       )
  #       experiment1_ds1[model_name]["best_inner_mcc"].append(
  #           best_inner_mcc
  #       )

  #       ################# Retrain on the outer training fold #################
  #       best_pipeline.fit(ds1_X_train, ds1_y_train)

  #       ################# Evaluate fold #################
  #       fold_metrics, ds1_y_pred, ds1_y_prob = evaluate_outer_fold(
  #           best_pipeline, ds1_X_test, ds1_y_test, threshold
  #           )

  #       fold_metrics['Fold'] = fold
  #       fold_metrics['Threshold'] = threshold
  #       experiment1_ds1[model_name]["fold_metrics"].append(
  #           fold_metrics
  #       )

  #       ################# Save fold path #################
  #       fold_path = save_fold_predictions(FOLD_DIR, model_name, fold, best_pipeline, best_params,
  #                                         threshold, test_idx, ds1_y_test, ds1_y_prob, ds1_y_pred)
  #       experiment1_ds1[model_name]["saved_folds"].append(
  #           fold_path
  #       )

  #       ################# Save trained model #################
  #       model_path = os.path.join(MODEL_DIR, f"{model_name}_fold{fold}.joblib")

  #       joblib.dump(best_pipeline, model_path)
  #       experiment1_ds1[model_name]["saved_models"].append(
  #           model_path
  #       )